# SOFR, FX & Cross-Currency Curves · Episode 5 — FX forwards and swaps

Companion notebook for the video: FX forwards and FX swaps on AUD/USD, forward points, covered interest parity, and the Australian dollar rate that FX swaps imply, compared with AONIA. Every number on screen or in the narration is produced here and read from `build/outputs.json`.

> **Data.** All quotes are illustrative, made up for teaching: AUD/USD spot and FX swap points (`quotes_fx_illustrative.csv`), SOFR swaps and futures (`quotes_sofr_illustrative.csv`, Episode 3), and AONIA swaps (`quotes_illustrative.csv`, the AUD series). They are shaped like a plausible market, not market data. Educational material only, not investment advice. Rates with Bip is not affiliated with the New York Fed.

1. Setup · 2. An FX swap · 3. Quotes · 4. Covered interest parity · 5. The implied AUD rate · 6. In QuantLib · 7. Export

**Running in Google Colab?** Run the next cells first: they install QuantLib (version 1.43, the one used in the video) and write the data files this notebook reads. Then run the rest of the notebook in order.

In [ ]:
# Colab doesn't include QuantLib. Install it (about 30 seconds).
# The video used QuantLib 1.43; drop '==1.43' for the latest.
!pip install QuantLib==1.43

In [ ]:
#@title Data: writes `quotes_fx_illustrative.csv` (run me first) { display-mode: "form" }
# Illustrative quotes, made up for teaching. Not market data.
from pathlib import Path
Path('quotes_fx_illustrative.csv').parent.mkdir(parents=True, exist_ok=True)
Path('quotes_fx_illustrative.csv').write_text("""instrument,label,value,unit
fx,spot,0.71,rate
fx,ON,-0.098,pips
fx,TN,-0.033,pips
fx,1W,-0.23,pips
fx,1M,-1.1,pips
fx,2M,-2.66,pips
fx,3M,-4.29,pips
fx,6M,-10.68,pips
fx,9M,-19.65,pips
fx,1Y,-30.32,pips
xccy,2Y,14.75,bp
xccy,3Y,14.75,bp
xccy,4Y,15.25,bp
xccy,5Y,15.5,bp
xccy,7Y,15.5,bp
xccy,10Y,15.5,bp
""")
print('wrote quotes_fx_illustrative.csv')

In [ ]:
#@title Data: writes `quotes_sofr_illustrative.csv` (run me first) { display-mode: "form" }
# Illustrative quotes, made up for teaching. Not market data.
from pathlib import Path
Path('quotes_sofr_illustrative.csv').parent.mkdir(parents=True, exist_ok=True)
Path('quotes_sofr_illustrative.csv').write_text("""instrument,label,value,unit
ois,1W,3.88,pct
ois,1M,3.87,pct
ois,2M,3.86,pct
ois,3M,3.85,pct
future,SR3 Dec 2026,96.20,price
future,SR3 Mar 2027,96.30,price
future,SR3 Jun 2027,96.39,price
future,SR3 Sep 2027,96.44,price
ois,2Y,3.66,pct
ois,3Y,3.63,pct
ois,5Y,3.63,pct
ois,7Y,3.70,pct
ois,10Y,3.81,pct
ois,15Y,3.94,pct
ois,20Y,3.98,pct
ois,30Y,3.96,pct
""")
print('wrote quotes_sofr_illustrative.csv')

In [ ]:
#@title Data: writes `quotes_illustrative.csv` (run me first) { display-mode: "form" }
# Illustrative quotes, made up for teaching. Not market data.
from pathlib import Path
Path('quotes_illustrative.csv').parent.mkdir(parents=True, exist_ok=True)
Path('quotes_illustrative.csv').write_text("""curve,instrument,tenor,value,unit
AONIA,deposit,O/N,3.85,pct
AONIA,OIS,1M,3.86,pct
AONIA,OIS,3M,3.89,pct
AONIA,OIS,6M,3.93,pct
AONIA,OIS,9M,3.97,pct
AONIA,OIS,1Y,4.00,pct
AONIA,OIS,18M,4.05,pct
AONIA,OIS,2Y,4.08,pct
AONIA,OIS,3Y,4.13,pct
AONIA,OIS,5Y,4.24,pct
AONIA,OIS,7Y,4.35,pct
AONIA,OIS,10Y,4.50,pct
BBSW3M,fixing,3M,4.02,pct
BBSW3M,AONIA/BBSW basis,1Y,13.0,bp
BBSW3M,AONIA/BBSW basis,2Y,14.0,bp
BBSW3M,AONIA/BBSW basis,3Y,15.0,bp
BBSW3M,AONIA/BBSW basis,5Y,15.5,bp
BBSW3M,AONIA/BBSW basis,7Y,16.0,bp
BBSW3M,AONIA/BBSW basis,10Y,16.5,bp
BBSW6M,fixing,6M,4.14,pct
BBSW6M,3s6s basis,1Y,8.0,bp
BBSW6M,3s6s basis,2Y,9.0,bp
BBSW6M,3s6s basis,3Y,10.0,bp
BBSW6M,3s6s basis,5Y,11.0,bp
BBSW6M,3s6s basis,7Y,11.5,bp
BBSW6M,3s6s basis,10Y,12.0,bp
""")
print('wrote quotes_illustrative.csv')

In [ ]:
# Parameters (papermill overrides these)
valuation_date = "2026-09-25"
fx_quotes_file = "quotes_fx_illustrative.csv"
sofr_quotes_file = "quotes_sofr_illustrative.csv"
aud_quotes_file = "quotes_illustrative.csv"
output_json = "build/outputs.json"

## 1. Setup: the SOFR curve (Episode 3) and the AONIA curve (the AUD series)

In [ ]:
import json
import datetime as dt
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import QuantLib as ql

today = ql.DateParser.parseISO(valuation_date)
ql.Settings.instance().evaluationDate = today
cal, dc = ql.Australia(), ql.Actual365Fixed()    # used by the AUD curve code
iso = lambda d: d.ISO()
pydate = lambda d: dt.date(d.year(), d.month(), d.dayOfMonth())
out = {"meta": {"series": "SOFR, FX & Cross-Currency Curves", "episode": 5, "valuation_date": valuation_date,
                "quantlib_version": ql.__version__, "quotes_label": "illustrative (made up for teaching)",
                "generated_at": dt.datetime.now().isoformat(timespec="seconds")}}

In [ ]:
# AUD curve family used from Episode 6 on. Copied verbatim into each notebook by make_notebook.py,
# so every notebook runs on its own. Needs: ql, pd, today, cal, dc (defined in the setup cell).

def aonia_index(curve=ql.YieldTermStructureHandle()):
    return ql.OvernightIndex("AONIA", 0, ql.AUDCurrency(), cal, dc, curve)

def bbsw(months, curve=ql.YieldTermStructureHandle()):
    # Set on the first day of each period (no fixing lag), Modified Following, no end-of-month rule.
    return ql.IborIndex(f"BBSW{months}M", ql.Period(months, ql.Months), 0, ql.AUDCurrency(),
                        cal, ql.ModifiedFollowing, False, dc, curve)

def build_curves(quotes):
    """AONIA from OIS quotes; 3M BBSW = AONIA + AONIA/BBSW basis; 6M BBSW = 3M BBSW + 3s6s basis.

    Returns the curves, their handles and the SimpleQuote behind every input, keyed (curve, tenor).
    Changing a quote with setValue() flows through all three curves.
    """
    q = {}
    def handle(row):
        scale = 1e4 if row.unit == "bp" else 100
        q[(row.curve, row.tenor)] = ql.SimpleQuote(row.value / scale)
        return ql.QuoteHandle(q[(row.curve, row.tenor)])

    rows = lambda curve: quotes[quotes.curve == curve].itertuples()
    MF = ql.ModifiedFollowing

    ois_helpers = []
    for r in rows("AONIA"):
        if r.instrument == "deposit":
            h = ql.DepositRateHelper(handle(r), ql.Period(1, ql.Days), 0, cal,
                                     ql.Following, False, dc)
        else:
            h = ql.OISRateHelper(1, ql.Period(r.tenor), handle(r), aonia_index(), paymentLag=2,
                                 paymentFrequency=ql.Annual, paymentCalendar=cal,
                                 convention=MF, endOfMonth=False)
        ois_helpers.append(h)
    aonia = ql.PiecewiseLogLinearDiscount(today, ois_helpers, dc)
    aonia.enableExtrapolation()
    aonia_h = ql.YieldTermStructureHandle(aonia)

    h3 = []
    for r in rows("BBSW3M"):
        if r.instrument == "fixing":
            h3.append(ql.DepositRateHelper(handle(r), bbsw(3)))
        else:  # AONIA + spread vs 3M BBSW, both quarterly
            h3.append(ql.OvernightIborBasisSwapRateHelper(
                handle(r), ql.Period(r.tenor), 1, cal, MF, False,
                aonia_index(aonia_h), bbsw(3), aonia_h))
    bbsw3m = ql.PiecewiseLogLinearDiscount(today, h3, dc)
    bbsw3m.enableExtrapolation()
    bbsw3m_h = ql.YieldTermStructureHandle(bbsw3m)

    h6 = []
    for r in rows("BBSW6M"):
        if r.instrument == "fixing":
            h6.append(ql.DepositRateHelper(handle(r), bbsw(6)))
        else:  # 3M BBSW + spread (quarterly) vs 6M BBSW (semi-annual)
            h6.append(ql.IborIborBasisSwapRateHelper(
                handle(r), ql.Period(r.tenor), 1, cal, MF, False,
                bbsw(3, bbsw3m_h), bbsw(6), aonia_h, False))
    bbsw6m = ql.PiecewiseLogLinearDiscount(today, h6, dc)
    bbsw6m.enableExtrapolation()

    helpers = {"AONIA": ois_helpers, "BBSW3M": h3, "BBSW6M": h6}
    curves = {"AONIA": aonia, "BBSW3M": bbsw3m, "BBSW6M": bbsw6m}
    for c in curves.values():
        c.nodes()  # bootstrap now, in order
    handles = {k: ql.YieldTermStructureHandle(c) for k, c in curves.items()}
    return curves, handles, q, helpers

def vanilla_swap(tenor, fixed_rate, index_months, forecast, discount, notional,
                 receive=True, start=None):
    """AUD vanilla swap: quarterly vs 3M BBSW or semi-annual vs 6M BBSW, ACT/365F, T+1 start."""
    start = start or cal.advance(today, 1, ql.Days)
    end = cal.advance(start, ql.Period(tenor), ql.ModifiedFollowing, False)
    freq = ql.Quarterly if index_months == 3 else ql.Semiannual
    sched = ql.Schedule(start, end, ql.Period(freq), cal, ql.ModifiedFollowing,
                        ql.ModifiedFollowing, ql.DateGeneration.Forward, False)
    side = ql.VanillaSwap.Receiver if receive else ql.VanillaSwap.Payer
    index = bbsw(index_months, forecast)
    sw = ql.VanillaSwap(side, notional, sched, fixed_rate, dc, sched, index, 0.0, dc)
    sw.setPricingEngine(ql.DiscountingSwapEngine(discount))
    return sw

In [ ]:
# SOFR curve used from Series 2 Episode 3 on. Copied verbatim into each notebook by make_notebook.py,
# so every notebook runs on its own. Needs: ql, pd, today (the evaluation date) from the setup cell.
#   SOFR OIS: CFTC Swaps Made Available to Trade chart (USD SOFR OIS): spot T+2, annual fixed and floating
#     legs, ACT/360, payment lag 2 days, New York business days; SOFR fixes on the government securities calendar.
#   3M SOFR futures: CME Submission 18-069: SOFR compounded over the IMM-to-IMM Reference Quarter,
#     price = 100 - rate; named by the month the Reference Quarter starts. Convexity set to zero (a simplification).
USD_CAL = ql.UnitedStates(ql.UnitedStates.Settlement)  # stand-in for New York business days
USD_DC = ql.Actual360()
MONTHS = {m: i for i, m in enumerate(["Jan", "Feb", "Mar", "Apr", "May", "Jun",
                                      "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"], start=1)}

def sofr_ois_helper(tenor, quote):
    return ql.OISRateHelper(2, ql.Period(tenor), quote, ql.Sofr(), paymentLag=2,
                            paymentFrequency=ql.Annual, paymentCalendar=USD_CAL,
                            convention=ql.ModifiedFollowing, endOfMonth=True)

def build_sofr_curve(quotes):
    """quotes: rows (instrument, label, value); OIS values in %, futures as prices. Returns curve, quotes, helpers."""
    q, helpers = {}, {}
    for r in quotes.itertuples():
        if r.instrument == "future":  # label like "SR3 Dec 2026"
            q[r.label] = ql.SimpleQuote(r.value)
            _, mon, year = r.label.split()
            helpers[r.label] = ql.SofrFutureRateHelper(ql.QuoteHandle(q[r.label]), MONTHS[mon], int(year),
                                                       ql.Quarterly)
        else:
            q[r.label] = ql.SimpleQuote(r.value / 100)
            helpers[r.label] = sofr_ois_helper(r.label, ql.QuoteHandle(q[r.label]))
    curve = ql.PiecewiseLogLinearDiscount(today, list(helpers.values()), USD_DC)
    curve.enableExtrapolation()
    curve.nodes()  # bootstrap now: the helpers only see the curve once it has been built
    return curve, q, helpers

In [ ]:
aud_curves, aud_handles, _, _ = build_curves(pd.read_csv(aud_quotes_file))
aonia = aud_handles["AONIA"]
sofr_curve, _, _ = build_sofr_curve(pd.read_csv(sofr_quotes_file))
sofr = ql.YieldTermStructureHandle(sofr_curve)
joint = ql.JointCalendar(ql.Australia(ql.Australia.Settlement), USD_CAL)
tom, spot = joint.advance(today, 1, ql.Days), joint.advance(today, 2, ql.Days)
fxq = pd.read_csv(fx_quotes_file)
S = float(fxq[fxq.label == "spot"].value.iloc[0])
print("today", today, "| tomorrow", tom, "| spot", spot, "| spot rate", S)

## 2. An FX swap

An FX swap exchanges two currencies on one date and reverses the exchange on a later date, at a rate agreed today (BIS Triennial Survey guidelines; RBA *Bulletin*, June 2010). Example: an Australian bank needs US dollars for three months. It gives AUD 100 million at spot and gets US dollars; in three months it gives the US dollars back and gets its Australian dollars back, at the three-month forward rate.

In [ ]:
points = {r.label: r.value for r in fxq[fxq.instrument == "fx"].itertuples() if r.label != "spot"}
def value_date(t):
    return joint.advance(spot, ql.Period(t), ql.ModifiedFollowing, True)
out["market"] = {"fx_swaps_share_pct": 42}  # RBA Bulletin, Feb 2026: "FX swaps ... 42 per cent of turnover"
aud_n = 100_000_000
d3 = value_date("3M")
F3 = S + points["3M"] / 1e4
out["swap_example"] = {
    "aud": aud_n, "near_date": iso(spot), "far_date": iso(d3), "spot": S, "forward": F3,
    "near_usd": aud_n * S, "far_usd": aud_n * F3,
    "rows": [
        {"leg": "Near (spot)", "date": iso(spot), "aud": f"pay AUD {aud_n:,.0f}", "usd": f"receive USD {aud_n * S:,.2f}"},
        {"leg": "Far (3 months)", "date": iso(d3), "aud": f"receive AUD {aud_n:,.0f}", "usd": f"pay USD {aud_n * F3:,.2f}"},
    ],
}
pd.DataFrame(out["swap_example"]["rows"])

## 3. Quotes

FX swaps and forwards are quoted as forward points: the forward rate minus the spot rate, in units of the fourth decimal place (QuantLib's `FxSwapRateHelper`: fwdFx = spotFx + fwdPoint). Overnight (today to tomorrow) and tom-next (tomorrow to spot) swaps cover the days before spot.

In [ ]:
qrows = [{"tenor": "Spot", "date": iso(spot), "points": 0.0, "outright": S}]
for t in ["1W", "1M", "2M", "3M", "6M", "9M", "1Y"]:
    qrows.append({"tenor": t, "date": iso(value_date(t)), "points": points[t], "outright": S + points[t] / 1e4})
out["quotes"] = {"spot": S, "rows": qrows, "on": points["ON"], "tn": points["TN"],
                 "today_rate": S - (points["TN"] + points["ON"]) / 1e4, "tom": iso(tom),
                 "y1_points": points["1Y"], "y1_outright": S + points["1Y"] / 1e4}
pd.DataFrame(qrows)

## 4. Covered interest parity

Borrowing US dollars directly should cost the same as borrowing Australian dollars and swapping them into US dollars (RBA *Bulletin*, June 2010). For AUD/USD (US dollars per Australian dollar), from spot $s$ to $T$:

$$F = S \times \frac{1 + r_\text{USD}\,\tau_{360}}{1 + r_\text{AUD}\,\tau_{365}} = S \times \frac{P_\text{AUD}(s,T)}{P_\text{USD}(s,T)}$$

With SOFR for the US dollar and AONIA for the Australian dollar:

In [ ]:
def growth(h, a, b):
    return h.discount(a) / h.discount(b)

T = d3
days = T - spot
r_usd = (growth(sofr, spot, T) - 1) * 360 / days
r_aonia = (growth(aonia, spot, T) - 1) * 365 / days
F_cip = S * growth(sofr, spot, T) / growth(aonia, spot, T)
out["cip3m"] = {"days": days, "sofr_pct": r_usd * 100, "aonia_pct": r_aonia * 100,
                "forward_cip": F_cip, "points_cip": (F_cip - S) * 1e4,
                "forward_market": F3, "points_market": points["3M"],
                "gap_pips": (F3 - F_cip) * 1e4}
pd.Series(out["cip3m"])

## 5. The Australian dollar rate the FX swap implies

Turn the formula around: given the market forward and SOFR, which Australian dollar rate makes parity hold?

$$r_\text{AUD}^\text{implied} = \left[\frac{S}{F}\,(1 + r_\text{USD}\,\tau_{360}) - 1\right]\frac{365}{\text{days}}$$

The gap between that rate and AONIA is the basis (RBA *Bulletin*, June 2010; BIS *Quarterly Review*, September 2016).

In [ ]:
imp = []
for t in ["1W", "1M", "2M", "3M", "6M", "9M", "1Y"]:
    d = value_date(t)
    n = d - spot
    F = S + points[t] / 1e4
    r_impl = ((S / F) * growth(sofr, spot, d) - 1) * 365 / n
    r_a = (growth(aonia, spot, d) - 1) * 365 / n
    imp.append({"tenor": t, "days": n, "sofr_pct": (growth(sofr, spot, d) - 1) * 360 / n * 100,
                "aonia_pct": r_a * 100, "implied_aud_pct": r_impl * 100, "basis_bp": (r_impl - r_a) * 1e4})
imp = pd.DataFrame(imp)
three = imp[imp.tenor == "3M"].iloc[0]
out["implied"] = {"rows": imp.to_dict("records"),
                  "m3_implied_pct": float(three.implied_aud_pct), "m3_aonia_pct": float(three.aonia_pct),
                  "m3_basis_bp": float(three.basis_bp),
                  "basis_min_bp": float(imp.basis_bp.min()), "basis_max_bp": float(imp.basis_bp.max())}
out["chart"] = {"months": [n / 365 * 12 for n in imp.days], "aonia_pct": imp.aonia_pct.tolist(),
                "implied_pct": imp.implied_aud_pct.tolist(), "sofr_pct": imp.sofr_pct.tolist(),
                "ticks": [{"x": m, "label": f"{m}M"} for m in (0, 3, 6, 9, 12)]}
imp

## 6. In QuantLib

`FxSwapRateHelper` bootstraps the Australian dollar curve that reproduces the forward points, given the US dollar (SOFR) curve. The overnight swap uses `fixingDays=0` and the tom-next swap `fixingDays=1`, both with a one-day tenor; their near rates are today's and tomorrow's rates, built back from spot (QuantLib documentation). The flag `isFxBaseCurrencyCollateralCurrency=False`: AUD is the base currency of AUD/USD, and the US dollar curve is the one we hold fixed. Why that matters is Episode 7.

In [ ]:
near_tn = ql.SimpleQuote(S - points["TN"] / 1e4)
near_on = ql.SimpleQuote(S - (points["TN"] + points["ON"]) / 1e4)
spot_q = ql.SimpleQuote(S)
helpers = [
    ql.FxSwapRateHelper(ql.QuoteHandle(ql.SimpleQuote(points["ON"] / 1e4)), ql.QuoteHandle(near_on),
                        ql.Period(1, ql.Days), 0, joint, ql.Following, False, False, sofr),
    ql.FxSwapRateHelper(ql.QuoteHandle(ql.SimpleQuote(points["TN"] / 1e4)), ql.QuoteHandle(near_tn),
                        ql.Period(1, ql.Days), 1, joint, ql.Following, False, False, sofr),
] + [ql.FxSwapRateHelper(ql.QuoteHandle(ql.SimpleQuote(points[t] / 1e4)), ql.QuoteHandle(spot_q),
                         ql.Period(t), 2, joint, ql.ModifiedFollowing, True, False, sofr)
     for t in ["1W", "1M", "2M", "3M", "6M", "9M", "1Y"]]
aud_fx = ql.PiecewiseLogLinearDiscount(today, helpers, dc)
aud_fx.nodes()
err = max(abs(h.impliedQuote() - h.quote().value()) * 1e4 for h in helpers)
d = value_date("3M")
out["quantlib"] = {"n_helpers": len(helpers), "max_error_pips": err,
                   "m3_implied_pct": (aud_fx.discount(spot) / aud_fx.discount(d) - 1) * 365 / (d - spot) * 100}
print("largest repricing error (pips):", err)
pd.Series(out["quantlib"])

## 7. Export for the video

In [ ]:
path = Path(output_json)
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(json.dumps(out, indent=2, default=float))
print("wrote", path.resolve(), f"({path.stat().st_size / 1024:.0f} KB)")